# Tools, Agents, and Persistence (Modern APIs)

This notebook covers tool calling, `create_agent`, and state persistence using a LangGraph checkpointer.

**Goals**
- Define tools with `@tool`.
- Bind tools to a model and inspect tool calls.
- Use `create_agent` for the tool loop.
- Persist conversation state by reusing the same `thread_id` (no deprecated history wrappers).

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openrouter import ChatOpenRouter
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.utils.uuid import uuid7

OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
print(f"Using OpenRouter model: {OPENROUTER_MODEL}")

model = ChatOpenRouter(model=OPENROUTER_MODEL, temperature=0.2)

Using OpenRouter model: google/gemini-3.5-flash


## Define tools
Tools are just functions with metadata. Keep signatures simple and explicit.

In [3]:
@tool
def simple_math(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '2+2*3'."""
    try:
        allowed = set("0123456789+-*/(). " )
        if any(ch not in allowed for ch in expression):
            return "Unsupported characters in expression."
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as exc:
        return f"Error: {exc}"

@tool
def lookup_definition(term: str) -> str:
    """Return a short definition for a small fixed glossary."""
    glossary = {
        "lcel": "LangChain Expression Language for composing runnables.",
        "retriever": "A runnable that returns Documents for a query.",
        "tool calling": "A model capability to request function/tool execution with structured args.",
    }
    return glossary.get(term.lower(), "No definition found.")

## Tool calling on the model
Bind tools and inspect the `tool_calls` in the AI message.

If your selected OpenRouter model does not support tool calling, this may return no tool calls.

In [ ]:
model_with_tools = model.bind_tools([simple_math, lookup_definition])
msg = model_with_tools.invoke("What is LCEL? Use the glossary tool.")
msg

In [ ]:
msg.tool_calls

## Agent loop + persistence (thread_id)
`create_agent` runs a model-tool loop.

To persist state across turns, pass a `checkpointer` and reuse a stable `thread_id` in `config`.

In [4]:
checkpointer = InMemorySaver()
agent = create_agent(
    model=model,
    tools=[simple_math, lookup_definition],
    system_prompt="You are a concise assistant. Use tools when helpful.",
    checkpointer=checkpointer,
)

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Taylor."}]},
    config=config,
)

{'messages': [HumanMessage(content='My name is Taylor.', additional_kwargs={}, response_metadata={}, id='ccb76905-cf8a-47bd-af16-c9b3eb5ac630'),
  AIMessage(content='Nice to meet you, Taylor! How can I help you today?', additional_kwargs={'reasoning_details': [{'data': 'AY89a1+NvOlC7ZfgomCOMD/2T6bYrpRiJk4lM9K1e+37JetST+1dyye9aPJXe2sNcugAg86BtL5fqnUU7T8JCzJV5SFZJvqhWmXPjItG2QxHmwIrPBybQ4/q5daf2jYDdRYsFfV0iOIEUALAWPQAgW4zfDIYq0jkMsxM/zLq8WKmW4ju7KSyM4Igec8Q3kZzX5mGMRsXVB1/wiHcYwzQ09tU7PnQf/Ym/H1WaJPvQg8+bNraxp/D6WVZVmG9JOj7ZPVq9Afr3pU8DQ65FVj/vvmWpCHeiMoQznw8ZX5p6AmHG4Aue1lvK0EfcKki52x0jM7taQp5Rv3QBvaSyFeDTDBg8P4jOiD4ncdB9oYzsHaFa/yoIko4GJoS7ZLLsbuHnJV3oTsz2FqkZJKgAe7tq2+2Uv7BRUBjTSv/OBXVz4FiSRERsHMJwfVxaSq+B7liQD3E38Ay38aenZGWah1f895IGyu/D7Td/YJIpMFfwAubJuQGb9povNUfFkzjeT1dClfQuQUFdjaK1RUOLREieUrm6pCmQ7oejm0w3ubwunRK6dpvaq4d/EaHdkawCXuqT84zEo903u1xeh7Rmqlq5CcgapasZXUH41Gvkx07umWgE5efZQIv66Bir9fd+VHNwDSfwkZP9trmbpjUx6+DdaBP7fZImlQLVhySn9sCAT1cqCG6xGTUtVmGwvwFoGtsivbpkDHA9YH1c9Eb+wcaKNxeb

In [5]:
# Same thread_id -> previous messages are available to the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config,
)

{'messages': [HumanMessage(content='My name is Taylor.', additional_kwargs={}, response_metadata={}, id='ccb76905-cf8a-47bd-af16-c9b3eb5ac630'),
  AIMessage(content='Nice to meet you, Taylor! How can I help you today?', additional_kwargs={'reasoning_details': [{'data': 'AY89a1+NvOlC7ZfgomCOMD/2T6bYrpRiJk4lM9K1e+37JetST+1dyye9aPJXe2sNcugAg86BtL5fqnUU7T8JCzJV5SFZJvqhWmXPjItG2QxHmwIrPBybQ4/q5daf2jYDdRYsFfV0iOIEUALAWPQAgW4zfDIYq0jkMsxM/zLq8WKmW4ju7KSyM4Igec8Q3kZzX5mGMRsXVB1/wiHcYwzQ09tU7PnQf/Ym/H1WaJPvQg8+bNraxp/D6WVZVmG9JOj7ZPVq9Afr3pU8DQ65FVj/vvmWpCHeiMoQznw8ZX5p6AmHG4Aue1lvK0EfcKki52x0jM7taQp5Rv3QBvaSyFeDTDBg8P4jOiD4ncdB9oYzsHaFa/yoIko4GJoS7ZLLsbuHnJV3oTsz2FqkZJKgAe7tq2+2Uv7BRUBjTSv/OBXVz4FiSRERsHMJwfVxaSq+B7liQD3E38Ay38aenZGWah1f895IGyu/D7Td/YJIpMFfwAubJuQGb9povNUfFkzjeT1dClfQuQUFdjaK1RUOLREieUrm6pCmQ7oejm0w3ubwunRK6dpvaq4d/EaHdkawCXuqT84zEo903u1xeh7Rmqlq5CcgapasZXUH41Gvkx07umWgE5efZQIv66Bir9fd+VHNwDSfwkZP9trmbpjUx6+DdaBP7fZImlQLVhySn9sCAT1cqCG6xGTUtVmGwvwFoGtsivbpkDHA9YH1c9Eb+wcaKNxeb

## Streaming an agent
Use `agent.stream(..., stream_mode=["messages", "updates"], version="v2")` to get both token chunks and completed node updates (model/tool messages).

In [6]:
from langchain.messages import AIMessage, AIMessageChunk, ToolMessage

input_state = {"messages": [{"role": "user", "content": "Compute (10+5)*2 and explain briefly."}]}

for chunk in agent.stream(
    input_state,
    config=config,
    stream_mode=["messages", "updates"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk) and token.content:
            print(token.content, end="", flush=True)
    elif chunk["type"] == "updates":
        for node_name, update in chunk["data"].items():
            if "messages" not in update:
                continue
            last = update["messages"][-1]
            if isinstance(last, AIMessage) and last.tool_calls:
                print(f"\n[tool_calls@{node_name}] {last.tool_calls}\n")
            if isinstance(last, ToolMessage):
                print(f"\n[tool_result@{node_name}] {last.content}\n")

print()


[tool_calls@model] [{'name': 'simple_math', 'args': {'expression': '(10+5)*2'}, 'id': 'tool_simple_math_Aj6AciFHZmMPyMJzuFvP', 'type': 'tool_call'}]


[tool_result@tools] 30

The result is **30**. 

**Explanation:**
1. First, solve the addition inside the parentheses: $10 + 5 = 15$.
2. Then, multiply the result by 2: $15 \times 2 = 30$.


## Common misuses to watch for
- Forgetting a checkpointer, then expecting `thread_id` to persist anything.
- Using a new `thread_id` each turn (you will lose continuity).
- Expecting tool calls from a model that does not support tool calling.
- Using agent loops when a simple LCEL chain is enough.